# Khám phá sơ bộ: Phân bố Giá và Mối quan hệ Giá – Điểm Review

**Mục tiêu:** Trực quan hóa hai đặc trưng cốt lõi của tập dữ liệu `wine_cleaned_final.csv` phục vụ báo cáo tiến trình:
1. **Phân bố `price`:** Khảo sát hình dạng phân bố, độ lệch và các giá trị cực trị của biến giá.
2. **Mối quan hệ `price` – `points`:** Khảo sát mức độ phân tán của điểm review theo giá trên dữ liệu quan sát thực tế (phân tích mô tả sơ bộ, không sử dụng đường hồi quy/kiểm định mô hình).

> **Quy tắc chọn mẫu:** Phân tích thực hiện trên các mẫu có giá quan sát thực tế trong tập dữ liệu sạch (`price_was_missing == False`). Không xóa các giá trị ngoại lai để đảm bảo tính trung thực của dữ liệu.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn nạp dữ liệu thống nhất
root = Path.cwd()
if root.name == 'w5_progress_parts':
    root = root.parent.parent
elif root.name == 'notebooks':
    root = root.parent

DATA_PATH = root / 'data' / 'processed' / 'wine_cleaned_final.csv'
df_all = pd.read_csv(DATA_PATH)

# Lọc mẫu có giá quan sát thực tế (theo quy tắc làm sạch dữ liệu)
if 'price_was_missing' in df_all.columns:
    df = df_all[df_all['price_was_missing'] == False].copy()
else:
    df = df_all.dropna(subset=['price']).copy()

n_total = len(df_all)
n_obs = len(df)
n_missing = n_total - n_obs

print(f'Tổng số dòng trong tập dữ liệu sạch: {n_total:,}')
print(f'Số mẫu có giá quan sát (mẫu phân tích): {n_obs:,}')
print(f'Số mẫu thiếu giá (được giữ lại để kiểm toán dữ liệu): {n_missing:,}')

## 1. Phân bố `price`

- **Số mẫu phân tích:** $N = 111,593$ mẫu quan sát có giá (loại trừ $8,395$ mẫu thiếu giá).
- **Trục tọa độ & Đơn vị:**
  - Trục hoành ($X$): Giá rượu (Đơn vị: USD, hiển thị dải chính 0–300 USD kèm ghi chú giá cao nhất tới 2,500 USD).
  - Trục tung ($Y$): Số lượng mẫu rượu quan sát (Đơn vị: Chai).
- **Phương pháp thể hiện:** Histogram (100 bins) kết hợp đường ước lượng mật độ hạt nhân KDE (Kernel Density Estimation) và 2 vạch mốc Trung vị (Median) và Trung bình (Mean).

In [ ]:
sns.set_theme(style='whitegrid', font_scale=1.05)

fig, ax1 = plt.subplots(figsize=(11, 6))
price_data = df['price'].dropna()

# 1. Vẽ Histogram cho dải giá phổ biến (0 - 300 USD)
ax1.hist(
    price_data,
    bins=100,
    range=(0, 300),
    color='#722F37',
    alpha=0.75,
    edgecolor='white',
    linewidth=0.5,
    label='Tần suất (Histogram, bins=100)'
)

# 2. Đường KDE trên trục phụ
ax1_kde = ax1.twinx()
sns.kdeplot(
    price_data,
    ax=ax1_kde,
    color='#C5A02E',
    linewidth=2.5,
    clip=(0, 300),
    label='Đường mật độ KDE'
)
ax1_kde.set_ylabel('')
ax1_kde.set_yticks([])

# 3. Mốc thống kê trung vị và trung bình
median_val = price_data.median()
mean_val = price_data.mean()
max_val = price_data.max()

ax1.axvline(median_val, color='#1E88E5', linestyle='--', linewidth=2, label=f'Trung vị (Median) = ${median_val:.0f}')
ax1.axvline(mean_val, color='#FB8C00', linestyle='--', linewidth=2, label=f'Trung bình (Mean) = ${mean_val:.1f}')

# Nhãn trục và tiêu đề
ax1.set_xlabel('Giá rượu (Đơn vị: USD)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Số lượng mẫu rượu (Chai)', fontsize=12, fontweight='bold')
ax1.set_title('Hình 1: Phân Bố Giá Rượu Vang (Wine Price Distribution)', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlim(0, 305)

# Khung nhận xét trực tiếp trên hình
note_text_1 = (
    f'• Số mẫu: N = {n_obs:,} chai có giá quan sát\n'
    f'• Trục X: Giá (USD) | Trục Y: Số lượng chai\n'
    f'• Dải giá: $4 – ${max_val:,.0f} USD (hiển thị trục đến $300)\n'
    f'• Trung vị = ${median_val:.0f} USD | Trung bình = ${mean_val:.1f} USD\n'
    f'• Nhận xét: Phân bố bị lệch phải rất mạnh (right-skewed).\n'
    f'  Đa số chai tập trung ở phân khúc phổ thông dưới $30.\n'
    f'  Không loại bỏ outlier để giữ tính khách quan, nhưng cần\n'
    f'  biến đổi logarit hoặc phân nhóm khi xây dựng mô hình.'
)
ax1.text(
    0.98, 0.95, note_text_1,
    transform=ax1.transAxes,
    fontsize=9.5,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='#FFFFF0', edgecolor='#BDBDBD', alpha=0.92)
)

ax1.legend(loc='center right', fontsize=10)
plt.tight_layout()
plt.show()

### Nhận xét tự trình bày (Hình 1):
1. **Mẫu và phạm vi:** Có $111,593$ chai có giá quan sát (dải giá từ \$4 đến \$2,500 USD). $8,395$ mẫu thiếu giá được giữ riêng theo quy trình kiểm soát dữ liệu sạch, không tự ý điền trung bình/trung vị.
2. **Hình dạng phân bố:** Dữ liệu có mức độ lệch phải (right-skewness) rất lớn. Nửa số chai rượu trong tập dữ liệu có giá từ \$25 USD trở xuống (Median = \$25), nhưng các dòng vang cao cấp đẩy giá trị trung bình lên \$35.6 USD.
3. **Ý nghĩa phân tích:** Toàn bộ các chai giá cao được giữ nguyên (không xóa outlier) để phản ánh đúng thực tế thị trường; việc giá lệch phải đặt ra yêu cầu kỹ thuật phải chuyển đổi sang thang logarit (`log_price`) trước khi thực hiện các phép hồi quy tuyến tính.

## 2. Quan hệ Giá – Điểm Review (`price` vs `points`)

- **Số mẫu phân tích:** $N = 111,593$ mẫu quan sát.
- **Trục tọa độ & Đơn vị:**
  - Trục hoành ($X$): Giá rượu (Đơn vị: USD, thang đo 0–300 USD).
  - Trục tung ($Y$): Điểm đánh giá của chuyên gia (Đơn vị: Points, thang điểm chuẩn từ 80 đến 100 điểm).
- **Phương pháp thể hiện:** Scatter plot với kỹ thuật giảm độ đậm ($lpha = 0.3$) và ánh xạ dải màu theo thang điểm để hiển thị rõ mật độ tại các vùng bị chồng lấp (overplotting). **Tuyệt đối không vẽ đường hồi quy/dự đoán** để giữ đúng tính chất khám phá mô tả sơ bộ ban đầu.

In [ ]:
fig, ax2 = plt.subplots(figsize=(11, 6))

# Để biểu đồ hiển thị rõ nét mật độ mà không bị lag do vẽ 111k điểm vector cùng lúc,
# ta lấy mẫu ngẫu nhiên có cố định seed (15,000 điểm đại diện đồng nhất)
np.random.seed(42)
if len(df) > 15000:
    df_plot = df.sample(15000, random_state=42)
else:
    df_plot = df

scatter = ax2.scatter(
    df_plot['price'],
    df_plot['points'],
    c=df_plot['points'],
    cmap='viridis',
    alpha=0.32,
    s=15,
    edgecolors='none'
)

# Thiết lập giới hạn và nhãn trục
ax2.set_xlabel('Giá rượu (Đơn vị: USD)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Điểm review (Đơn vị: Points, thang 80–100)', fontsize=12, fontweight='bold')
ax2.set_title('Hình 2: Quan Hệ Giữa Giá và Điểm Review (Price vs. Points Scatter Plot)', fontsize=14, fontweight='bold', pad=15)
ax2.set_xlim(0, 305)
ax2.set_ylim(79, 101)

# Thêm thanh màu thang điểm
cbar = fig.colorbar(scatter, ax=ax2, pad=0.02, shrink=0.88)
cbar.set_label('Thang điểm (Points)', fontsize=10.5)

# Khung nhận xét trực tiếp trên hình
note_text_2 = (
    f'• Số mẫu: N = {n_obs:,} chai quan sát (hiển thị sample 15,000 điểm để rõ mật độ)\n'
    f'• Trục X: Giá (USD) | Trục Y: Điểm review (Points 80–100)\n'
    f'• Nhận xét khám phá sơ bộ:\n'
    f'  - Tồn tại xu hướng đồng biến: Rượu giá cao hầu như không có điểm thấp (< 85).\n'
    f'  - Độ phân tán ở phân khúc phổ thông ($10 – $50) là rất lớn: cùng một mức giá\n'
    f'    có thể đạt điểm từ 80 đến trên 95.\n'
    f'  - Kết luận: Giá mang thông tin nhưng CHƯA ĐỦ để giải thích điểm review.\n'
    f'    Cần thêm metadata (quốc gia, giống nho, vùng trồng) ở các bước tiếp theo.'
)
ax2.text(
    0.98, 0.05, note_text_2,
    transform=ax2.transAxes,
    fontsize=9.5,
    verticalalignment='bottom',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='#FFFFF0', edgecolor='#BDBDBD', alpha=0.92)
)

plt.tight_layout()
plt.show()

### Nhận xét tự trình bày (Hình 2):
1. **Mật độ và xu hướng cận dưới:** Mật độ điểm dày đặc nhất ở dải giá dưới \$50 USD. Đáng chú ý là sự xuất hiện của một "sàn điểm" di động theo giá: rượu giá càng cao thì điểm tối thiểu càng tăng (hầu như không có chai nào trên \$100 mà bị chấm dưới 85 điểm).
2. **Độ phân tán phương sai:** Trong cùng một mức giá phổ biến (ví dụ quanh \$20–\$30 USD), điểm số dao động rất rộng từ 80 đến tận 95+ điểm. Điều này cho thấy giá trị của một chai rượu không quy định tuyệt đối điểm số mà ban giám khảo đánh giá.
3. **Kết luận phục vụ đề tài:** Biểu đồ mô tả rõ ràng rằng *chỉ dùng giá thì chưa thể giải thích trọn vẹn điểm review*. Đây là bằng chứng trực quan củng cố giả thuyết của nhóm: cần đưa thêm thông tin sản phẩm (metadata: giống nho, xuất xứ, niên vụ) để giải thích phần chênh lệch điểm số này.